In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
import mlflow
import mlflow.sklearn

df = pd.read_csv("../data/processed/reviews_clean.csv")
df = df.dropna(subset=['clean_content'])  # jaga-jaga kalau ada hasil cleaning yang kosong

X = df['clean_content']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(y_train.value_counts())

Train: 226, Test: 57
sentiment
positive    141
negative     85
Name: count, dtype: int64


In [2]:
mlflow.set_experiment("sentiment-ewallet")

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

with mlflow.start_run(run_name="baseline_tfidf_logreg"):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    report = classification_report(y_test, y_pred, output_dict=True)
    print(classification_report(y_test, y_pred))

    # Log parameter yang dipakai
    mlflow.log_param("max_features", 3000)
    mlflow.log_param("ngram_range", "(1,2)")
    mlflow.log_param("model", "LogisticRegression")

    # Log metrik utama
    mlflow.log_metric("accuracy", report['accuracy'])
    mlflow.log_metric("f1_positive", report['positive']['f1-score'])
    mlflow.log_metric("f1_negative", report['negative']['f1-score'])
    mlflow.log_metric("recall_negative", report['negative']['recall'])

    # Simpan model
    mlflow.sklearn.log_model(pipeline, "model")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.75      0.95      0.84        22
    positive       0.97      0.80      0.88        35

    accuracy                           0.86        57
   macro avg       0.86      0.88      0.86        57
weighted avg       0.88      0.86      0.86        57



2026/09/08 09:34:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Confusion Matrix:
[[21  1]
 [ 7 28]]


In [3]:
import joblib

joblib.dump(pipeline, "../models/sentiment_pipeline.joblib")
print("Model tersimpan di models/sentiment_pipeline.joblib")

Model tersimpan di models/sentiment_pipeline.joblib
